# Object Detection Toolkit - GEMASTIK Competition
## Synthetic CIFAR-10 Object Detection Dataset

This notebook provides a complete pipeline for object detection training and evaluation using synthetic datasets generated from CIFAR-10.

### Competition Phases:
- **Phase 0**: Environment Setup
- **Phase 1**: Data Generation & Preparation
- **Phase 2**: Baseline Model Training
- **Phase 3**: Core Modeling (Speed & Accuracy Tracks)
- **Phase 4**: Ensemble with WBF
- **Phase 5**: Final Evaluation, XAI, and Prediction

---
## 🔧 Phase 0: Pre-Competition Preparation (Your Toolkit)

### Setup your environment and template scripts

In [ ]:
# Install required packages
# Uncomment and run if packages are not installed
# !pip install -r requirements.txt

In [ ]:
# Import all necessary libraries
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from pathlib import Path
from tqdm.notebook import tqdm

# Deep learning frameworks
import torch
from torchvision import datasets, transforms
from ultralytics import YOLO

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n✓ Environment setup complete!")

In [ ]:
# Configuration
CONFIG = {
    'canvas_size': 640,
    'object_size': 32,
    'num_train': 10000,
    'num_val': 2000,
    'min_objects': 1,
    'max_objects': 5,
    'cifar_root': './data',
    'output_root': './synthetic_dataset',
    'seed': SEED
}

# CIFAR-10 class names
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

---
## 🚀 Phase 1: Data Generation & Preparation (Critical!)

This is the most important phase - we're creating the problem!

### 1.1. EDA (Object Inventory Analysis)

In [ ]:
# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset for EDA...")
transform = transforms.ToTensor()
cifar_train = datasets.CIFAR10(root=CONFIG['cifar_root'], train=True, 
                               download=True, transform=transform)
cifar_test = datasets.CIFAR10(root=CONFIG['cifar_root'], train=False, 
                              download=True, transform=transform)

print(f"\nTraining samples: {len(cifar_train)}")
print(f"Test samples: {len(cifar_test)}")
print(f"Number of classes: {len(CLASS_NAMES)}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
# Analyze class distribution
labels = [cifar_train[i][1] for i in range(len(cifar_train))]
label_counts = pd.Series(labels).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
axes[0].bar(range(10), label_counts.values, color='steelblue')
axes[0].set_xticks(range(10))
axes[0].set_xticklabels(CLASS_NAMES, rotation=45)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_title('CIFAR-10 Class Distribution (Training Set)')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(label_counts.values, labels=CLASS_NAMES, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (Percentage)')

plt.tight_layout()
plt.show()

print("\n✓ Dataset is balanced - each class has equal representation")

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for class_id in range(10):
    # Find first image of this class
    idx = labels.index(class_id)
    img, label = cifar_train[idx]
    
    # Convert to numpy and display
    img_np = img.permute(1, 2, 0).numpy()
    axes[class_id].imshow(img_np)
    axes[class_id].set_title(CLASS_NAMES[class_id])
    axes[class_id].axis('off')

plt.suptitle('Sample Images from Each CIFAR-10 Class (32x32 pixels)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nImage shape: {img.shape} (C, H, W)")
print(f"All images are 32x32 pixels - these will be our 'objects'")

### 1.2. Synthetic Dataset Generation (Required)

Generate dataset: (large_image, list_of_bounding_boxes) from CIFAR-10 assets

In [ ]:
# Option 1: Use the provided script
# !python generate_synthetic_dataset.py

# Option 2: Generate dataset in the notebook (recommended for interactive exploration)
from generate_synthetic_dataset import SyntheticDatasetGenerator

# Initialize generator
generator = SyntheticDatasetGenerator(
    cifar_root=CONFIG['cifar_root'],
    output_root=CONFIG['output_root'],
    canvas_size=CONFIG['canvas_size'],
    object_size=CONFIG['object_size'],
    seed=CONFIG['seed']
)

print("Synthetic Dataset Generator initialized!")

In [ ]:
# Generate a sample synthetic image to visualize
print("Generating sample synthetic image...")
canvas, labels = generator.generate_image(cifar_train, min_objects=3, max_objects=5)

# Display the canvas
plt.figure(figsize=(10, 10))
plt.imshow(canvas)
plt.title(f'Sample Synthetic Image ({CONFIG["canvas_size"]}x{CONFIG["canvas_size"]} with {len(labels)} objects)')
plt.axis('off')
plt.show()

print(f"\nGenerated {len(labels)} objects on canvas")
print("\nLabels (YOLO format: class_id x_center y_center width height):")
for i, label in enumerate(labels[:5]):  # Show first 5
    parts = label.split()
    class_id = int(parts[0])
    print(f"  Object {i+1}: {CLASS_NAMES[class_id]} - {label}")
if len(labels) > 5:
    print(f"  ... and {len(labels) - 5} more objects")

In [ ]:
# Generate full dataset (this will take some time)
print("Starting full dataset generation...")
print("This may take 10-20 minutes depending on your system.")
print("\nYou can reduce num_train and num_val for faster generation during testing.")

generator.generate_dataset(
    num_train=CONFIG['num_train'],
    num_val=CONFIG['num_val'],
    min_objects=CONFIG['min_objects'],
    max_objects=CONFIG['max_objects']
)

print("\n✓ Dataset generation complete!")

In [ ]:
# Verify generated dataset
train_images = list(Path(CONFIG['output_root']) / 'train' / 'images').glob('*.png')
train_labels = list((Path(CONFIG['output_root']) / 'train' / 'labels').glob('*.txt'))
val_images = list((Path(CONFIG['output_root']) / 'val' / 'images').glob('*.png'))
val_labels = list((Path(CONFIG['output_root']) / 'val' / 'labels').glob('*.txt'))

print("Dataset Verification:")
print(f"  Training images: {len(train_images)}")
print(f"  Training labels: {len(train_labels)}")
print(f"  Validation images: {len(val_images)}")
print(f"  Validation labels: {len(val_labels)}")
print(f"\n  Data config file: {Path(CONFIG['output_root']) / 'data.yaml'}")

# Analyze objects per image
objects_per_image = []
for label_file in train_labels:
    with open(label_file, 'r') as f:
        objects_per_image.append(len(f.readlines()))

plt.figure(figsize=(10, 5))
plt.hist(objects_per_image, bins=range(1, CONFIG['max_objects']+2), edgecolor='black', alpha=0.7)
plt.xlabel('Number of Objects per Image')
plt.ylabel('Frequency')
plt.title('Distribution of Objects per Image in Training Set')
plt.xticks(range(1, CONFIG['max_objects']+1))
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"\nAverage objects per image: {np.mean(objects_per_image):.2f}")
print(f"Min objects: {np.min(objects_per_image)}")
print(f"Max objects: {np.max(objects_per_image)}")

### 1.3. Data Augmentation

**Good news!** Modern YOLO frameworks (ultralytics) have powerful built-in augmentation:
- Mosaic
- MixUp
- Flip
- HSV shift

No need for manual albumentations during training - just enable in configuration!

---
## 🏃 Phase 2: Baseline Model (Hours 2-4)

Get the first mAP (mean Average Precision) score on the leaderboard

### 2.1. Baseline Training

In [ ]:
# Training configuration for baseline
BASELINE_CONFIG = {
    'data': str(Path(CONFIG['output_root']) / 'data.yaml'),
    'model': 'yolov8m.pt',  # YOLOv8-M: good balance of speed and accuracy
    'epochs': 25,
    'batch': 16,
    'imgsz': 640,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'project': 'runs/train',
    'name': 'baseline_yolov8m',
    'patience': 10,
    'save': True,
    'plots': True
}

print("Baseline Training Configuration:")
for key, value in BASELINE_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Option 1: Train using command line (uncomment to use)
# !yolo train data={BASELINE_CONFIG['data']} model={BASELINE_CONFIG['model']} \
#     epochs={BASELINE_CONFIG['epochs']} batch={BASELINE_CONFIG['batch']} \
#     imgsz={BASELINE_CONFIG['imgsz']} device={BASELINE_CONFIG['device']} \
#     project={BASELINE_CONFIG['project']} name={BASELINE_CONFIG['name']}

# Option 2: Train using Python API (recommended for notebooks)
from train import train_yolo

print("Starting baseline training...")
print("This will take 30-60 minutes depending on your GPU.")

results = train_yolo(**BASELINE_CONFIG)

print("\n✓ Baseline training complete!")

### 2.2. Baseline Evaluation & First Submission

In [ ]:
# Load trained model for evaluation
baseline_model_path = Path(BASELINE_CONFIG['project']) / BASELINE_CONFIG['name'] / 'weights' / 'best.pt'
print(f"Loading baseline model from: {baseline_model_path}")

baseline_model = YOLO(str(baseline_model_path))

# Validate on validation set
val_results = baseline_model.val(data=BASELINE_CONFIG['data'])

print("\nBaseline Model Performance:")
print(f"  mAP@0.5: {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall: {val_results.box.mr:.4f}")

In [ ]:
# Visualize training results
results_dir = Path(BASELINE_CONFIG['project']) / BASELINE_CONFIG['name']

# Display results plots if available
results_img = results_dir / 'results.png'
if results_img.exists():
    img = Image.open(results_img)
    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results - Baseline Model')
    plt.show()
else:
    print("Results plot not found. Check the training directory.")

In [ ]:
# Make predictions on validation set for submission
val_images_dir = Path(CONFIG['output_root']) / 'val' / 'images'

print("Running inference on validation set...")
baseline_predictions = baseline_model.predict(
    source=str(val_images_dir),
    conf=0.25,
    save=True,
    project='runs/predict',
    name='baseline_predictions'
)

print("\n✓ Baseline predictions complete!")

In [ ]:
# Convert predictions to submission format
from inference import results_to_csv

baseline_df = results_to_csv(
    baseline_predictions,
    output_path='baseline_submission.csv',
    class_names=CLASS_NAMES
)

print("\nSubmission Preview:")
print(baseline_df.head(10))
print(f"\nTotal predictions: {len(baseline_df)}")
print(f"\nSubmission file ready: baseline_submission.csv")

---
## 🏋️ Phase 3: Core Modeling (Parallel Experiments)

Run multiple experiments in parallel to find the best model

### 3.1. Track: Extreme Speed

**Goal**: Fastest inference performance, possibly with slightly lower mAP

In [ ]:
# Configuration for speed-optimized model
SPEED_CONFIG = {
    'data': str(Path(CONFIG['output_root']) / 'data.yaml'),
    'model': 'yolov8n.pt',  # YOLOv8-Nano: fastest variant
    'epochs': 25,
    'batch': 32,  # Can use larger batch for smaller model
    'imgsz': 640,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'project': 'runs/train',
    'name': 'speed_yolov8n',
    'patience': 10
}

print("Speed Track Configuration:")
for key, value in SPEED_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Train speed-optimized model
print("Training speed-optimized model (YOLOv8-Nano)...")
speed_results = train_yolo(**SPEED_CONFIG)
print("\n✓ Speed model training complete!")

# Evaluate
speed_model_path = Path(SPEED_CONFIG['project']) / SPEED_CONFIG['name'] / 'weights' / 'best.pt'
speed_model = YOLO(str(speed_model_path))
speed_val = speed_model.val(data=SPEED_CONFIG['data'])

print("\nSpeed Model Performance:")
print(f"  mAP@0.5: {speed_val.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {speed_val.box.map:.4f}")
print(f"  Inference speed: ~{speed_val.speed['inference']:.1f}ms per image")

### 3.2. Track: Maximum Accuracy

**Goal**: Highest mAP possible, speed is not a concern

In [ ]:
# Configuration for accuracy-optimized model
ACCURACY_CONFIG = {
    'data': str(Path(CONFIG['output_root']) / 'data.yaml'),
    'model': 'yolov8x.pt',  # YOLOv8-XLarge: most accurate variant
    'epochs': 50,  # More epochs for better convergence
    'batch': 8,   # Smaller batch due to larger model
    'imgsz': 640,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'project': 'runs/train',
    'name': 'accuracy_yolov8x',
    'patience': 15
}

print("Accuracy Track Configuration:")
for key, value in ACCURACY_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Train accuracy-optimized model
print("Training accuracy-optimized model (YOLOv8-XLarge)...")
print("⚠️ Warning: This will take significantly longer (1-2 hours)")
accuracy_results = train_yolo(**ACCURACY_CONFIG)
print("\n✓ Accuracy model training complete!")

# Evaluate
accuracy_model_path = Path(ACCURACY_CONFIG['project']) / ACCURACY_CONFIG['name'] / 'weights' / 'best.pt'
accuracy_model = YOLO(str(accuracy_model_path))
accuracy_val = accuracy_model.val(data=ACCURACY_CONFIG['data'])

print("\nAccuracy Model Performance:")
print(f"  mAP@0.5: {accuracy_val.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {accuracy_val.box.map:.4f}")
print(f"  Inference speed: ~{accuracy_val.speed['inference']:.1f}ms per image")

In [ ]:
# Optional: Train RT-DETR (Transformer-based detector)
RTDETR_CONFIG = {
    'data': str(Path(CONFIG['output_root']) / 'data.yaml'),
    'model': 'rtdetr-x.pt',  # RT-DETR-XLarge
    'epochs': 50,
    'batch': 8,
    'imgsz': 640,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'project': 'runs/train',
    'name': 'accuracy_rtdetr_x',
    'patience': 15
}

print("RT-DETR Configuration (Transformer Architecture):")
for key, value in RTDETR_CONFIG.items():
    print(f"  {key}: {value}")

# Uncomment to train RT-DETR
# rtdetr_results = train_yolo(**RTDETR_CONFIG)
# rtdetr_model_path = Path(RTDETR_CONFIG['project']) / RTDETR_CONFIG['name'] / 'weights' / 'best.pt'
# rtdetr_model = YOLO(str(rtdetr_model_path))
# rtdetr_val = rtdetr_model.val(data=RTDETR_CONFIG['data'])

### Compare All Models

In [ ]:
# Compare model performance
comparison_data = {
    'Model': ['YOLOv8-M (Baseline)', 'YOLOv8-N (Speed)', 'YOLOv8-X (Accuracy)'],
    'mAP@0.5': [
        val_results.box.map50,
        speed_val.box.map50,
        accuracy_val.box.map50
    ],
    'mAP@0.5:0.95': [
        val_results.box.map,
        speed_val.box.map,
        accuracy_val.box.map
    ],
    'Inference Speed (ms)': [
        val_results.speed['inference'],
        speed_val.speed['inference'],
        accuracy_val.speed['inference']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# mAP comparison
x = np.arange(len(comparison_df))
width = 0.35
axes[0].bar(x - width/2, comparison_df['mAP@0.5'], width, label='mAP@0.5', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['mAP@0.5:0.95'], width, label='mAP@0.5:0.95', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('mAP')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Speed comparison
axes[1].bar(range(len(comparison_df)), comparison_df['Inference Speed (ms)'], alpha=0.8, color='coral')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Inference Time (ms)')
axes[1].set_title('Model Speed Comparison (Lower is Better)')
axes[1].set_xticks(range(len(comparison_df)))
axes[1].set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 🤝 Phase 4: Ensemble (Advanced Technique)

**Goal**: Combine predictions from multiple models using Weighted Boxes Fusion (WBF)

**Why WBF?** It intelligently merges overlapping boxes and combines confidence scores, much better than standard NMS.

In [ ]:
# Setup ensemble configuration
from ensemble_wbf import EnsembleWBF

# Select best models for ensemble
ensemble_models = [
    str(baseline_model_path),    # YOLOv8-M
    str(accuracy_model_path),    # YOLOv8-X
]

# Assign weights (can be based on validation mAP)
ensemble_weights = [
    val_results.box.map,      # Baseline weight
    accuracy_val.box.map,     # Accuracy model weight
]

print("Ensemble Configuration:")
for i, (model, weight) in enumerate(zip(ensemble_models, ensemble_weights)):
    print(f"  Model {i+1}: {Path(model).parent.parent.name} (weight: {weight:.4f})")

# Create ensemble predictor
ensemble = EnsembleWBF(
    model_paths=ensemble_models,
    weights=ensemble_weights,
    iou_thr=0.5,
    skip_box_thr=0.0001,
    conf_type='avg'
)

print("\n✓ Ensemble predictor initialized!")

In [ ]:
# Test ensemble on a single image
test_image = list((Path(CONFIG['output_root']) / 'val' / 'images').glob('*.png'))[0]

print(f"Testing ensemble on: {test_image.name}")
boxes, scores, labels = ensemble.predict_single_image(str(test_image))

print(f"\nEnsemble detected {len(boxes)} objects")

# Visualize
img = cv2.imread(str(test_image))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

for box, score, label in zip(boxes, scores, labels):
    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(img, f"{CLASS_NAMES[int(label)]}: {score:.2f}", 
                (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

plt.figure(figsize=(12, 12))
plt.imshow(img)
plt.title('Ensemble Prediction (WBF)')
plt.axis('off')
plt.show()

In [ ]:
# Run ensemble on full validation set
print("Running ensemble predictions on validation set...")
print("This will take some time as it runs multiple models.")

ensemble_df = ensemble.predict_directory(
    image_dir=str(Path(CONFIG['output_root']) / 'val' / 'images'),
    output_csv='ensemble_submission.csv',
    imgsz=640,
    conf=0.001,
    visualize=False
)

print("\n✓ Ensemble predictions complete!")
print("\nEnsemble Submission Preview:")
print(ensemble_df.head(10))

---
## 🏁 Phase 5: Final Evaluation, XAI, and Prediction

Understand where the model fails and explain predictions

### 5.1. In-Depth Evaluation

In [ ]:
# Per-class performance analysis
from ultralytics.utils.metrics import ConfusionMatrix

# Get confusion matrix from best model
best_model = accuracy_model  # or use ensemble results

# Validate and get detailed metrics
val_metrics = best_model.val(data=ACCURACY_CONFIG['data'])

# Display per-class mAP
print("Per-Class Performance (mAP@0.5:0.95):")
print("="*50)
for i, class_name in enumerate(CLASS_NAMES):
    if hasattr(val_metrics.box, 'class_result'):
        map_class = val_metrics.box.class_result(i)[2]  # mAP@0.5:0.95
        print(f"  {class_name:12s}: {map_class:.4f}")

print("\nOverall Metrics:")
print(f"  mAP@0.5: {val_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_metrics.box.map:.4f}")
print(f"  Precision: {val_metrics.box.mp:.4f}")
print(f"  Recall: {val_metrics.box.mr:.4f}")

In [ ]:
# Visualize confusion matrix
confusion_matrix_path = Path(ACCURACY_CONFIG['project']) / ACCURACY_CONFIG['name'] / 'confusion_matrix.png'

if confusion_matrix_path.exists():
    img = Image.open(confusion_matrix_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix - Accuracy Model')
    plt.show()
    
    print("\nConfusion Matrix Insights:")
    print("- Diagonal values show correct classifications")
    print("- Off-diagonal values show misclassifications")
    print("- Look for similar classes being confused (e.g., cat vs dog, car vs truck)")
else:
    print("Confusion matrix not available")

In [ ]:
# Analyze failure cases
print("Analyzing potential failure modes:")
print("\n1. Class Confusion:")
print("   - Similar classes: cat/dog, automobile/truck, ship/airplane")
print("   - Check confusion matrix for high off-diagonal values")

print("\n2. Localization Issues:")
print("   - Box accuracy: Should be precise (32x32 objects)")
print("   - Check IoU distribution in validation results")

print("\n3. Missed Detections:")
print("   - Small/overlapping objects may be missed")
print("   - Check recall scores per class")

print("\n4. False Positives:")
print("   - Background detections")
print("   - Check precision scores per class")

### 5.2. Model Explainability (XAI)

Validate that the model focuses on objects, not background artifacts

In [ ]:
# Visualize predictions with attention
print("Explainability Analysis:")
print("\nWhat we want to see:")
print("  ✓ Model focuses on 32x32 object regions")
print("  ✓ Heatmaps highlight object areas")
print("  ✓ No attention on background")

print("\nWhat would indicate problems:")
print("  ✗ Attention spread across background")
print("  ✗ Heatmaps focus on canvas edges")
print("  ✗ Model using background patterns as shortcuts")

# Note: Full Grad-CAM implementation requires additional setup
# For competition, visual inspection of predictions is often sufficient
print("\n💡 Tip: Use YOLO's built-in prediction visualization to inspect:")
print("   - Confidence scores distribution")
print("   - Bounding box accuracy")
print("   - Detection consistency")

In [ ]:
# Sample prediction visualization with confidence analysis
sample_images = list((Path(CONFIG['output_root']) / 'val' / 'images').glob('*.png'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, img_path in enumerate(sample_images):
    # Predict
    results = best_model.predict(str(img_path), conf=0.25, verbose=False)
    
    # Load and visualize
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    if results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        confs = results[0].boxes.conf.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy().astype(int)
        
        for box, conf, cls in zip(boxes, confs, classes):
            x1, y1, x2, y2 = box.astype(int)
            # Color by confidence
            color = (0, int(255*conf), int(255*(1-conf)))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, f"{CLASS_NAMES[cls]}: {conf:.2f}", 
                       (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    
    axes[i].imshow(img)
    axes[i].set_title(f'Image {i+1}: {len(results[0].boxes) if results[0].boxes else 0} objects')
    axes[i].axis('off')

plt.suptitle('Sample Predictions (Color: Green=High Conf, Red=Low Conf)', fontsize=14)
plt.tight_layout()
plt.show()

### 5.3. Final Prediction Pipeline

In [ ]:
# Final submission workflow
print("Final Submission Checklist:")
print("\n1. Model Selection:")
print("   □ Single best model: YOLOv8-X")
print("   ☑ Ensemble (WBF): YOLOv8-M + YOLOv8-X")

print("\n2. Confidence Threshold:")
print("   - Current: 0.001 (very low to capture all detections)")
print("   - Can adjust based on precision/recall tradeoff")

print("\n3. Output Format:")
print("   - CSV with columns: image_id, class_id, confidence, x_min, y_min, x_max, y_max")
print("   - Sorted by image_id and confidence")

print("\n4. Files Generated:")
print("   ✓ baseline_submission.csv")
print("   ✓ ensemble_submission.csv")

print("\n5. Next Steps:")
print("   - Submit ensemble_submission.csv to leaderboard")
print("   - Monitor performance")
print("   - Iterate if needed")

In [ ]:
# Final statistics
print("\n" + "="*60)
print("FINAL COMPETITION SUMMARY")
print("="*60)

print("\nDataset Statistics:")
print(f"  Training images: {CONFIG['num_train']}")
print(f"  Validation images: {CONFIG['num_val']}")
print(f"  Classes: {len(CLASS_NAMES)}")
print(f"  Objects per image: {CONFIG['min_objects']}-{CONFIG['max_objects']}")

print("\nModels Trained:")
print(f"  1. Baseline (YOLOv8-M): mAP@0.5:0.95 = {val_results.box.map:.4f}")
print(f"  2. Speed (YOLOv8-N): mAP@0.5:0.95 = {speed_val.box.map:.4f}")
print(f"  3. Accuracy (YOLOv8-X): mAP@0.5:0.95 = {accuracy_val.box.map:.4f}")
print(f"  4. Ensemble (WBF): Expected improvement over single models")

print("\nSubmission Files:")
print(f"  - baseline_submission.csv ({len(baseline_df)} detections)")
print(f"  - ensemble_submission.csv ({len(ensemble_df)} detections)")

print("\n" + "="*60)
print("🎉 Competition Pipeline Complete! Good luck!")
print("="*60)

---
## 📊 Additional Analysis and Utilities

In [ ]:
# Export model for deployment
print("Model Export Options:")
print("\n1. ONNX (for cross-platform deployment):")
print("   best_model.export(format='onnx')")

print("\n2. TensorRT (for NVIDIA GPU optimization):")
print("   best_model.export(format='engine')")

print("\n3. CoreML (for iOS deployment):")
print("   best_model.export(format='coreml')")

# Example: Export to ONNX
# best_model.export(format='onnx', dynamic=True, simplify=True)

In [ ]:
# Benchmark inference speed
import time

def benchmark_model(model, num_images=100):
    """Benchmark model inference speed."""
    test_images = list((Path(CONFIG['output_root']) / 'val' / 'images').glob('*.png'))[:num_images]
    
    times = []
    for img_path in tqdm(test_images, desc="Benchmarking"):
        start = time.time()
        _ = model.predict(str(img_path), verbose=False)
        times.append(time.time() - start)
    
    return {
        'mean': np.mean(times) * 1000,  # ms
        'std': np.std(times) * 1000,
        'min': np.min(times) * 1000,
        'max': np.max(times) * 1000,
        'fps': 1 / np.mean(times)
    }

print("Benchmarking models (this will take a minute)...")
baseline_bench = benchmark_model(baseline_model)
speed_bench = benchmark_model(speed_model)
accuracy_bench = benchmark_model(accuracy_model)

print("\nInference Speed Comparison:")
print(f"\nBaseline (YOLOv8-M):")
print(f"  Mean: {baseline_bench['mean']:.2f} ms")
print(f"  FPS: {baseline_bench['fps']:.1f}")

print(f"\nSpeed (YOLOv8-N):")
print(f"  Mean: {speed_bench['mean']:.2f} ms")
print(f"  FPS: {speed_bench['fps']:.1f}")

print(f"\nAccuracy (YOLOv8-X):")
print(f"  Mean: {accuracy_bench['mean']:.2f} ms")
print(f"  FPS: {accuracy_bench['fps']:.1f}")

---
## 🎯 Competition Tips & Best Practices

### Key Takeaways:

1. **Data Quality > Model Complexity**
   - Ensure synthetic data is realistic
   - Verify label accuracy
   - Balance object distribution

2. **Ensemble is Powerful**
   - WBF significantly improves results
   - Combine diverse models (different architectures/scales)
   - Weight models by validation performance

3. **Hyperparameter Tuning**
   - Learning rate schedules
   - Augmentation intensity
   - Confidence thresholds
   - NMS/WBF IoU thresholds

4. **Monitoring & Analysis**
   - Track metrics per class
   - Analyze confusion patterns
   - Visualize predictions regularly
   - Use XAI to verify model behavior

5. **Competition Strategy**
   - Start simple (baseline)
   - Iterate quickly
   - Use ensemble for final submission
   - Monitor leaderboard feedback

### Common Pitfalls to Avoid:

- ❌ Overfitting to validation set
- ❌ Ignoring class imbalance
- ❌ Using too low confidence threshold
- ❌ Not verifying data quality
- ❌ Forgetting to set random seeds

### Time Management:

- **Hours 1-2**: Setup + Data Generation
- **Hours 3-4**: Baseline Model
- **Hours 5-8**: Parallel Model Training
- **Hours 9-10**: Ensemble + Final Submission

---

**Good luck with the GEMASTIK competition! 🚀**